### Setup

In [ ]:
!pip install transformers
!pip install datasets
!pip install evaluate
!pip install langdetect
!pip install nlpaug

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 15.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=fe2982911c267d62d475e506e9fd52a70579c27b037546872ef737efb5a245bd
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 10.3 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
import numpy as np
import torch
from sklearn.model_selection import train_test_split
import evaluate

In [ ]:
if torch.cuda.is_available():
  device = "cuda"
elif torch.backends.mps.is_available():
  device = "mps"
else:
  device = "cpu"
print(f"Using device: {device}")

Using device: cuda


In [ ]:
data_dir = "/content/drive/MyDrive/data/"
models_dir = "/content/drive/MyDrive/models/"

text_column_name = "description"
permission_name = "Microphone : record audio"
label_column_name = permission_name

model_name = "distilbert-base-uncased"
test_size = 0.2
num_labels = 2

In [ ]:
sorted_permissions = pd.read_csv(data_dir+'sorted_permissions.csv')

permissions = sorted_permissions['Permission'].head(30).tolist()
permissions

['Other : full network access',
 'Other : view network connections',
 'Other : prevent device from sleeping',
 'Photos/Media/Files : read the contents of your USB storage',
 'Storage : read the contents of your USB storage',
 'Uncategorized : receive data from Internet',
 'Photos/Media/Files : modify or delete the contents of your USB storage',
 'Storage : modify or delete the contents of your USB storage',
 'Other : control vibration',
 'Other : run at startup',
 'Camera : take pictures and videos',
 'Wi-Fi connection information : view Wi-Fi connections',
 'Location : precise location (GPS and network-based)',
 'Location : approximate location (network-based)',
 'Other : pair with Bluetooth devices',
 'Microphone : record audio',
 'Other : change your audio settings',
 'Other : access Bluetooth settings',
 'Calendar : read calendar events plus confidential information',
 "Calendar : add or modify calendar events and send email to guests without owners' knowledge",
 'Other : Google Pl

In [ ]:
df = pd.read_csv(data_dir+'/apps_with_permissions_cleaned_with_names_and_NEW_permissions.csv')
df.head()

,Calendar : add or modify calendar events and send email to guests without owners' knowledge,Calendar : read calendar events plus confidential information,Camera : take pictures and videos,Contacts : find accounts on the device,Contacts : modify your contacts,Contacts : read your contacts,Device & app history : read sensitive log data,Device & app history : read your Web bookmarks and history,Device & app history : retrieve running apps,Device & app history : retrieve system internal state,...,Uncategorized : write TV channel/program information,Uncategorized : write subscribed feeds,Wearable sensors/Activity data : body sensors (like heart rate monitors),Wi-Fi connection information : view Wi-Fi connections,appId,description,name,permissions,summary,url
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,homeworkout.homeworkouts.noequipment,Home Workouts provides daily workout routines ...,Home Workout - No Equipment,{'Photos/Media/Files': ['modify or delete the ...,Home workouts for fitness & fast bodybuilding ...,https://play.google.com/store/apps/details?id=...
1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,de.biotronik.PatientApp,The Patient App from BIOTRONIK SE & Co. KG sup...,Patient App,{'Location': ['precise location (GPS and netwo...,BIOTRONIK Patient App,https://play.google.com/store/apps/details?id=...
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,com.umpshealth,The Umps Health Smart Home uses a set of smart...,Umps Health,{'Other': ['full network access']},Umps Health supports seniors to live safely an...,https://play.google.com/store/apps/details?id=...
3,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,com.technogym.mywellness.fitnessmobil,With Fitness Mobilé you to get most out of the...,Mijn FMS,"{'Identity': ['add or remove accounts', 'find ...",With Fitness Mobilé you to get most out of the...,https://play.google.com/store/apps/details?id=...
4,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,com.trainerize.semperstronger,Take Semper Stronger with you wherever you go ...,MCHN Strength,"{'Calendar': [""add or modify calendar events a...",Leading fitness app.,https://play.google.com/store/apps/details?id=...


In [ ]:
# Top 30 permissions
df = df[permissions + ['appId', 'description', 'name', 'permissions', 'summary', 'url']]
df.head()

,Other : full network access,Other : view network connections,Other : prevent device from sleeping,Photos/Media/Files : read the contents of your USB storage,Storage : read the contents of your USB storage,Uncategorized : receive data from Internet,Photos/Media/Files : modify or delete the contents of your USB storage,Storage : modify or delete the contents of your USB storage,Other : control vibration,Other : run at startup,...,Phone : directly call phone numbers,Contacts : find accounts on the device,Identity : find accounts on the device,Other : connect and disconnect from Wi-Fi,appId,description,name,permissions,summary,url
0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,homeworkout.homeworkouts.noequipment,Home Workouts provides daily workout routines ...,Home Workout - No Equipment,{'Photos/Media/Files': ['modify or delete the ...,Home workouts for fitness & fast bodybuilding ...,https://play.google.com/store/apps/details?id=...
1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,de.biotronik.PatientApp,The Patient App from BIOTRONIK SE & Co. KG sup...,Patient App,{'Location': ['precise location (GPS and netwo...,BIOTRONIK Patient App,https://play.google.com/store/apps/details?id=...
2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,com.umpshealth,The Umps Health Smart Home uses a set of smart...,Umps Health,{'Other': ['full network access']},Umps Health supports seniors to live safely an...,https://play.google.com/store/apps/details?id=...
3,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,0.0,com.technogym.mywellness.fitnessmobil,With Fitness Mobilé you to get most out of the...,Mijn FMS,"{'Identity': ['add or remove accounts', 'find ...",With Fitness Mobilé you to get most out of the...,https://play.google.com/store/apps/details?id=...
4,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,com.trainerize.semperstronger,Take Semper Stronger with you wherever you go ...,MCHN Strength,"{'Calendar': [""add or modify calendar events a...",Leading fitness app.,https://play.google.com/store/apps/details?id=...


### Preprocessing

In [ ]:
# Check if app descriptions are in english
from langdetect import detect
def is_english(text):
    try:
        return detect(text) == 'en'
    except:
        return False

df = df[df['description'].apply(is_english)]

In [ ]:
# Preprocessing the app descriptions

# Removing special symbols that might b used for new points in the description
import re

def preprocess_description(text):
    text = re.sub(r'<.*?>', ' ', text) # Remove html tags like <b></b> etc.
    text = re.sub(r'\n', ' ', text)  # Replace newlines with space
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove special characters
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces

    return text


# Removing stopwords
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    words = text.split()
    filtered_words = [word for word in words if word.lower() not in stop_words]
    return ' '.join(filtered_words)

df['description'] = df['description'].apply(preprocess_description)
df['description'] = df['description'].apply(remove_stopwords)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
# Save the dataframe with top 30 permissions to a new CSV
df.to_csv(data_dir + 'apps_with_permissions_cleaned_with_names_and_NEW_permissions_TOP30.csv', index=False)
print("DataFrame with top 30 permissions saved.")

DataFrame with top 30 permissions saved.


### Sampling the dataset

In [ ]:
sample_df = df[['appId', 'description']+[permission_name]]
sample_df.head()

,appId,description,Microphone : record audio
0,homeworkout.homeworkouts.noequipment,Home Workouts provides daily workout routines ...,0.0
1,de.biotronik.PatientApp,Patient App BIOTRONIK SE Co KG supports patien...,0.0
2,com.umpshealth,Umps Health Smart Home uses set smart plugs fi...,0.0
3,com.technogym.mywellness.fitnessmobil,Fitness Mobil get services facility train indo...,1.0
4,com.trainerize.semperstronger,Take Semper Stronger wherever go Semper Strong...,1.0


In [ ]:
sample_df[permission_name] = sample_df[permission_name].astype(int)
sample_df.head()

/tmp/ipython-input-1333953604.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sample_df[permission_name] = sample_df[permission_name].astype(int)


,appId,description,Microphone : record audio
0,homeworkout.homeworkouts.noequipment,Home Workouts provides daily workout routines ...,0
1,de.biotronik.PatientApp,Patient App BIOTRONIK SE Co KG supports patien...,0
2,com.umpshealth,Umps Health Smart Home uses set smart plugs fi...,0
3,com.technogym.mywellness.fitnessmobil,Fitness Mobil get services facility train indo...,1
4,com.trainerize.semperstronger,Take Semper Stronger wherever go Semper Strong...,1


In [ ]:
sample_df = sample_df.rename(columns={text_column_name: 'text', label_column_name: 'label'})
sample_df.head()

,appId,text,label
0,homeworkout.homeworkouts.noequipment,Home Workouts provides daily workout routines ...,0
1,de.biotronik.PatientApp,Patient App BIOTRONIK SE Co KG supports patien...,0
2,com.umpshealth,Umps Health Smart Home uses set smart plugs fi...,0
3,com.technogym.mywellness.fitnessmobil,Fitness Mobil get services facility train indo...,1
4,com.trainerize.semperstronger,Take Semper Stronger wherever go Semper Strong...,1


In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
df_train, df_test = train_test_split(sample_df, test_size=test_size, random_state=42)

In [ ]:
train_dataset = Dataset.from_pandas(df_train)
test_dataset = Dataset.from_pandas(df_test)

### Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/4279 [00:00<?, ? examples/s]

Map:   0%|          | 0/1070 [00:00<?, ? examples/s]

### Initialize Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics

)

/tmp/ipython-input-1984462095.py:12: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

In [ ]:
trainer.save_model(models_dir+f'{permission_name}')

### Evaluate model

In [ ]:
from sklearn.metrics import classification_report

#### On train data

In [ ]:
preds = trainer.predict(tokenized_train)
preds = np.argmax(preds[:3][0],axis=1)
GT = df_train['label'].tolist()
print(classification_report(GT,preds))

              precision    recall  f1-score   support

           0       0.87      0.89      0.88      2079
           1       0.89      0.88      0.89      2199

    accuracy                           0.88      4278
   macro avg       0.88      0.88      0.88      4278
weighted avg       0.88      0.88      0.88      4278



#### On test data

In [ ]:
preds = trainer.predict(tokenized_test)
preds = np.argmax(preds[:3][0],axis=1) #preds[:3][1]
GT = df_test['label'].tolist()
print(classification_report(GT,preds))

              precision    recall  f1-score   support

           0       0.81      0.82      0.81       515
           1       0.83      0.82      0.82       555

    accuracy                           0.82      1070
   macro avg       0.82      0.82      0.82      1070
weighted avg       0.82      0.82      0.82      1070



In [ ]:
accuracy = sum(preds==GT)/len(GT)
print(f"Accuracy: {accuracy}")

Accuracy: 0.8186915887850468


#### Testing model on user input text

In [ ]:
new_text = [sample_df[sample_df['label'] == 0].iloc[1000]['text']]

# Tokenize the new text
inputs = tokenizer(new_text, return_tensors="pt", truncation=True, padding=True).to(device)

# Make predictions
with torch.no_grad():
    logits = model(**inputs).logits

# Get the predicted label
predicted_label_id = torch.argmax(logits, dim=1).item()

# Assuming 0 is negative and 1 is positive for your binary classification
predicted_label = "Positive" if predicted_label_id == 1 else "Negative"

print(f"New Text: {new_text[0]}")
print(f"Predicted Label ID: {predicted_label_id}")
print(f"Predicted Label: {predicted_label}")

New Text: Accupedo accurate pedometer app monitors daily walking easy read charts history logs monitor steps calories burned distance time best walking buddy Accupedo motivate walk Set daily goal step towards healthier Accupedo pedometer Features intelligent algorithm starts tracking 8 12 consecutive steps stops restarts automatically walk Daily log history step counts distance calories walking time Charts Daily weekly monthly yearly step counts Smart messages todays quote Power usage mode options efficient power saving Customized personal settings sensitivity unit metricenglish step distance body weight daily goal etc Concise widget display Home screen 1x1 4x1 Edit daily step counts Database backup Google Drive Widget skin colors black blue green orange pink transparent Share daily log Facebook Email daily log file Works intelligent 3D motion recognition algorithm embedded track walking patterns filtering ejecting nonwalking activities Accupedo counts steps regardless put phone like p

### Creating pipeline for all permissions

## ‼️Note: Changed the model on 11/04
From "distilbert-base-uncased" to "microsoft/deberta-v3-small". Also changed some directories in gdrive.

In [ ]:
!pip install transformers
!pip install datasets
!pip install evaluate
!pip install langdetect
!pip install nlpaug

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 19.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=5079a6c6d8af6848058a2e43763da1a6a3c8815c09708840236883db25db112e
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 9.9 MB/s eta 0:00:00


In [ ]:
# Run only if running from scratch

from google.colab import drive
drive.mount('/content/drive')

#
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
import numpy as np
import torch
from sklearn.model_selection import train_test_split
import evaluate
from sklearn.metrics import classification_report

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Using device: {device}")

# Params
data_dir = "/content/drive/MyDrive/data/"
# models_dir = "/content/drive/MyDrive/distilbert/models/"
models_dir = "/content/drive/MyDrive/deberta/models/"

text_column_name = "description"
permission_name = "Microphone : record audio"
label_column_name = permission_name

# model_name = "distilbert-base-uncased"
model_name = "microsoft/deberta-v3-base"
test_size = 0.2
num_labels = 2

#
sorted_permissions = pd.read_csv(data_dir+'sorted_permissions.csv')

permissions = sorted_permissions['Permission'].head(30).tolist()

#
df = pd.read_csv(data_dir+'/apps_with_permissions_cleaned_with_names_and_NEW_permissions.csv')

# Top 30 permissions
df = df[permissions + ['appId', 'description', 'name', 'permissions', 'summary', 'url']]

### Preprocessing
# Check if app descriptions are in english
from langdetect import detect
def is_english(text):
    try:
        return detect(text) == 'en'
    except:
        return False
df = df[df['description'].apply(is_english)]


# Preprocessing the app descriptions
# Removing special symbols that might b used for new points in the description
import re
def preprocess_description(text):
    text = re.sub(r'<.*?>', ' ', text) # Remove html tags like <b></b> etc.
    text = re.sub(r'\n', ' ', text)  # Replace newlines with space
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove special characters
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text

# Removing stopwords
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))
def remove_stopwords(text):
    words = text.split()
    filtered_words = [word for word in words if word.lower() not in stop_words]
    return ' '.join(filtered_words)

df['description'] = df['description'].apply(preprocess_description)
# Skip stopword removal for DeBERTa: comment below line
# df['description'] = df['description'].apply(remove_stopwords)

Mounted at /content/drive
Using device: cuda


In [ ]:
import nlpaug.augmenter.word as naw

def augment_data(df, permission_name):
    aug = naw.SynonymAug(aug_src='wordnet')

    class_counts = df[permission_name].value_counts()

    majority_label = class_counts.idxmax()
    minority_label = class_counts.idxmin()
    max_count = class_counts.max()

    minority_df = df[df[permission_name] == minority_label]
    needed = max_count - len(minority_df)

    if needed <= 0:
        return df

    n_per_sample = (needed + len(minority_df) - 1) // len(minority_df)

    augmented_texts = []
    for _, row in minority_df.iterrows():
        for _ in range(n_per_sample):
            if len(augmented_texts) >= needed:
                break
            try:
                augmented_text = aug.augment(row['text'])
            except Exception as e:
                augmented_text = row['text'] # Fallback if augmentation fails

            augmented_texts.append({
                'appId': f"aug_{row['appId']}",
                'text': augmented_text,
                'label': minority_label
            })
    augmented_df = pd.DataFrame(augmented_texts)

    return pd.concat([df, augmented_df], ignore_index=True)

In [ ]:
i = 0
def train_pipeline(permission):
    global i
    i += 1
    print(f"Training for permission: {i}_{permission}")
    global permission_name, label_column_name
    permission_name = permission
    label_column_name = permission_name

    sample_df = df[['appId', 'description']+[permission_name]]
    sample_df[permission_name] = sample_df[permission_name].astype(int)

    sample_df = sample_df.rename(columns={text_column_name: 'text', label_column_name: 'label'})

    # Using nlpaug to handle class imbalance
    sample_df = augment_data(sample_df, 'label')

    df_train, df_test = train_test_split(sample_df, test_size=test_size, random_state=42)

    train_dataset = Dataset.from_pandas(df_train)
    test_dataset = Dataset.from_pandas(df_test)

    ### Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def preprocess_function(examples):
        return tokenizer(examples["text"], truncation=True)

    tokenized_train = train_dataset.map(preprocess_function, batched=True)
    tokenized_test = test_dataset.map(preprocess_function, batched=True)



    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    metric = evaluate.load("accuracy")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        return metric.compute(predictions=predictions, references=labels)

    # distilbert
    # training_args = TrainingArguments(
    #     output_dir="./results",
    #     learning_rate=2e-4,
    #     per_device_train_batch_size=8,
    #     per_device_eval_batch_size=8,
    #     num_train_epochs=3,
    #     weight_decay=0.01,
    #     logging_strategy="epoch",
    #     report_to="none",
    # )

    # deberta
    training_args = TrainingArguments(
        output_dir="./results",
        learning_rate=3e-5,  # Lower learning rate (was 2e-4)
        per_device_train_batch_size=8,  # Can try 16 if memory allows
        per_device_eval_batch_size=8,
        num_train_epochs=5,  # More epochs (was 3)
        weight_decay=0.01,
        warmup_steps=500,  # Add warmup for better convergence
        logging_strategy="epoch",
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_test,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    trainer.train()

    trainer.save_model(models_dir+f"{i}_{permission_name.replace('/', '-')}")




    # report_filename = f"/content/drive/MyDrive/distilbert/reports/{i}_{permission_name.replace('/', '-')}_classification_report.txt"
    report_filename = f"/content/drive/MyDrive/deberta/reports/{i}_{permission_name.replace('/', '-')}_classification_report.txt"

    with open(report_filename, 'w') as f:
        f.write(f"Classification Report for {permission_name}:\n\n")

        # Training data
        preds = trainer.predict(tokenized_train)
        preds = np.argmax(preds[:3][0],axis=1)
        GT = df_train['label'].tolist()
        train_report = classification_report(GT,preds)

        # Test data
        preds = trainer.predict(tokenized_test)
        preds = np.argmax(preds[:3][0],axis=1)
        GT = df_test['label'].tolist()
        test_report = classification_report(GT, preds)

        f.write("Training Data Report:\n")
        f.write(train_report)
        f.write("\n\nTest Data Report:\n")
        f.write(test_report)

    print(f"Classification reports saved to {report_filename}")



In [ ]:
permissions

['Other : full network access',
 'Other : view network connections',
 'Other : prevent device from sleeping',
 'Photos/Media/Files : read the contents of your USB storage',
 'Storage : read the contents of your USB storage',
 'Uncategorized : receive data from Internet',
 'Photos/Media/Files : modify or delete the contents of your USB storage',
 'Storage : modify or delete the contents of your USB storage',
 'Other : control vibration',
 'Other : run at startup',
 'Camera : take pictures and videos',
 'Wi-Fi connection information : view Wi-Fi connections',
 'Location : precise location (GPS and network-based)',
 'Location : approximate location (network-based)',
 'Other : pair with Bluetooth devices',
 'Microphone : record audio',
 'Other : change your audio settings',
 'Other : access Bluetooth settings',
 'Calendar : read calendar events plus confidential information',
 "Calendar : add or modify calendar events and send email to guests without owners' knowledge",
 'Other : Google Pl

In [ ]:
i = 5 # initially i = 0, change as needed
for permission in permissions[i:]:
    # with open('/content/drive/MyDrive/distilbert/reports/log.txt', 'a') as f:
    with open('/content/drive/MyDrive/deberta/reports/log.txt', 'a') as f:
        f.write(f"Training for permission: {i+1}_{permission}\n")
    train_pipeline(permission)

Training for permission: 6_Uncategorized : receive data from Internet


Streaming output truncated to the last 5000 lines.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_dat

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


... (174 outputs left out)

Map:   0%|          | 0/2026 [00:00<?, ? examples/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-4115309533.py:70: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Step,Training Loss
1013,0.372100
2026,0.080600
3039,0.018700
4052,0.008300
5065,0.001600


Classification reports saved to /content/drive/MyDrive/deberta/reports/30_Other : connect and disconnect from Wi-Fi_classification_report.txt


### Error analysis

In [ ]:
!pip install transformers
!pip install datasets
!pip install evaluate
!pip install langdetect
!pip install nlpaug

In [ ]:
# Run only if running from scratch

from google.colab import drive
drive.mount('/content/drive')

#
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
import numpy as np
import torch
from sklearn.model_selection import train_test_split
import evaluate
from sklearn.metrics import classification_report

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Using device: {device}")

# Params
data_dir = "/content/drive/MyDrive/data/"
models_dir = "/content/drive/MyDrive/models/"

text_column_name = "description"
permission_name = "Microphone : record audio"
label_column_name = permission_name

model_name = "distilbert-base-uncased"
test_size = 0.2
num_labels = 2

#
sorted_permissions = pd.read_csv(data_dir+'sorted_permissions.csv')

permissions = sorted_permissions['Permission'].head(30).tolist()

#
df = pd.read_csv(data_dir+'/apps_with_permissions_cleaned_with_names_and_NEW_permissions.csv')

# Top 30 permissions
df = df[permissions + ['appId', 'description', 'name', 'permissions', 'summary', 'url']]

### Preprocessing
# Check if app descriptions are in english
from langdetect import detect
def is_english(text):
    try:
        return detect(text) == 'en'
    except:
        return False
df = df[df['description'].apply(is_english)]

# Create original_description column [ADDED FOR ERROR ANALYSIS]
df['original_description'] = df['description']

# Preprocessing the app descriptions
# Removing special symbols that might b used for new points in the description
import re
def preprocess_description(text):
    text = re.sub(r'<.*?>', ' ', text) # Remove html tags like <b></b> etc.
    text = re.sub(r'\n', ' ', text)  # Replace newlines with space
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove special characters
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text

# Removing stopwords
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))
def remove_stopwords(text):
    words = text.split()
    filtered_words = [word for word in words if word.lower() not in stop_words]
    return ' '.join(filtered_words)

df['description'] = df['description'].apply(preprocess_description)
df['description'] = df['description'].apply(remove_stopwords)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda


In [ ]:
import nlpaug.augmenter.word as naw

def augment_data(df, permission_name):
    aug = naw.SynonymAug(aug_src='wordnet')

    class_counts = df[permission_name].value_counts()

    majority_label = class_counts.idxmax()
    minority_label = class_counts.idxmin()
    max_count = class_counts.max()

    minority_df = df[df[permission_name] == minority_label]
    needed = max_count - len(minority_df)

    if needed <= 0:
        return df

    n_per_sample = (needed + len(minority_df) - 1) // len(minority_df)

    augmented_texts = []
    for _, row in minority_df.iterrows():
        for _ in range(n_per_sample):
            if len(augmented_texts) >= needed:
                break
            try:
                augmented_text = aug.augment(row['text'])
            except Exception as e:
                augmented_text = row['text'] # Fallback if augmentation fails

            augmented_texts.append({
                'appId': f"aug_{row['appId']}",
                'text': augmented_text,
                'label': minority_label
            })
    augmented_df = pd.DataFrame(augmented_texts)

    return pd.concat([df, augmented_df], ignore_index=True)

In [ ]:
i = 0
def error_analysis(permission):
    global i
    i += 1
    print(f"Error analysis for permission: {i}_{permission}")
    global permission_name, label_column_name
    permission_name = permission
    label_column_name = permission_name

    sample_df = df[['appId', 'description']+[permission_name]]
    sample_df[permission_name] = sample_df[permission_name].astype(int)

    sample_df = sample_df.rename(columns={text_column_name: 'text', label_column_name: 'label'})

    # Using nlpaug to handle class imbalance
    sample_df = augment_data(sample_df, 'label')

    df_train, df_test = train_test_split(sample_df, test_size=test_size, random_state=42)

    train_dataset = Dataset.from_pandas(df_train)
    test_dataset = Dataset.from_pandas(df_test)

    ### Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def preprocess_function(examples):
        return tokenizer(examples["text"], truncation=True)

    tokenized_train = train_dataset.map(preprocess_function, batched=True)
    tokenized_test = test_dataset.map(preprocess_function, batched=True)


    # Load the trained model
    model_path = models_dir + f"{i}_{permission_name.replace('/', '-')}"
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    print(f"Loaded model from {model_path}")

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    metric = evaluate.load("accuracy")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        return metric.compute(predictions=predictions, references=labels)

    training_args = TrainingArguments(
        output_dir="./results",
        learning_rate=2e-4,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=3,
        weight_decay=0.01,
        logging_strategy="epoch",
        report_to="none",
    )

    # Use a dummy trainer for prediction
    trainer = Trainer(
        model=model,
        args=training_args,
        eval_dataset=tokenized_test,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )


    # Perform predictions on the test set
    predictions = trainer.predict(tokenized_test)
    logits = predictions.predictions
    predicted_labels = np.argmax(logits, axis=-1)
    true_labels = tokenized_test["label"]

    # Identify misclassified rows
    misclassified_indices = np.where(predicted_labels != true_labels)[0]

    # Create a DataFrame with misclassified rows and predicted information
    misclassified_df = df_test.iloc[misclassified_indices].copy()
    misclassified_df['predicted_label'] = predicted_labels[misclassified_indices]
    misclassified_df['predicted_score'] = np.max(torch.softmax(torch.tensor(logits[misclassified_indices]), dim=1).numpy(), axis=1)

    # Drop the 'text' column (which is the preprocessed description)
    misclassified_df = misclassified_df.drop(columns=['text'])

    # Create a temporary column without "aug_" prefix for merging
    misclassified_df['original_appId'] = misclassified_df['appId'].apply(lambda x: x.replace('aug_', '') if x.startswith('aug_') else x)

    # Merge with the original dataframe to get the original_description
    misclassified_df = pd.merge(misclassified_df, df[['appId', 'original_description']], left_on='original_appId', right_on='appId', how='left')

    # Drop the redundant 'appId' and the temporary 'original_appId' columns
    misclassified_df = misclassified_df.drop(columns=['appId_y', 'original_appId'])
    misclassified_df = misclassified_df.rename(columns={'appId_x': 'appId'})


    # Sort misclassified data by predicted_score descending
    misclassified_df = misclassified_df.sort_values(by='predicted_score', ascending=False)

    # Keep only unique appIds rows
    misclassified_df = misclassified_df.drop_duplicates(subset=['appId'])

    # Save the misclassified data
    misclassified_filename = f"/content/drive/MyDrive/error_analysis/{i}_{permission_name.replace('/', '-')}_misclassified_data.csv"
    misclassified_df.to_csv(misclassified_filename, index=False)

    print(f"Misclassified data saved to {misclassified_filename}")

In [ ]:
i = 0 # initially i = 0, change as needed
for permission in permissions[i:]:
    with open('/content/drive/MyDrive/error_analysis/log.txt', 'a') as f:
        f.write(f"Error analysis for permission: {i+1}_{permission}\n")
    error_analysis(permission)

Error analysis for permission: 1_Other : full network access


Streaming output truncated to the last 5000 lines.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_dat

Map:   0%|          | 0/8494 [00:00<?, ? examples/s]

Map:   0%|          | 0/2124 [00:00<?, ? examples/s]

Loaded model from /content/drive/MyDrive/models/1_Other : full network access


/tmp/ipython-input-3857288188.py:59: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


... (171 outputs left out)

Map:   0%|          | 0/2026 [00:00<?, ? examples/s]

Loaded model from /content/drive/MyDrive/models/30_Other : connect and disconnect from Wi-Fi


/tmp/ipython-input-3857288188.py:59: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Misclassified data saved to /content/drive/MyDrive/error_analysis/30_Other : connect and disconnect from Wi-Fi_misclassified_data.csv
